<a href="https://colab.research.google.com/github/manaskesarla/Machine-learning-lab/blob/main/Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Program 10: Non-parametric methods - k-NN classifier and Kernel Density Estimation (KDE),
            with a performance comparison.

Topics covered:
  - k-NN classifier: effect of k, distance metrics, choosing k with cross-validation
  - Decision boundaries for different k (underfitting vs overfitting)
  - Kernel Density Estimation in 1-D: effect of kernel and bandwidth
  - Choosing bandwidth with cross-validation (GridSearchCV)
  - KDE-based Bayes classifier
  - Performance comparison: k-NN vs KDE classifier vs Gaussian Naive Bayes (parametric)
"""

# ------------------------------------------------------------
# 1. Import required libraries
# ------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.datasets import load_wine, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KernelDensity
from sklearn.naive_bayes import GaussianNB
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

# ============================================================
# PART A: k-NEAREST NEIGHBOURS CLASSIFIER
# ============================================================
print("=" * 60)
print("PART A: k-NN CLASSIFIER (Wine dataset)")
print("=" * 60)

# ------------------------------------------------------------
# 2. Load the Wine dataset (3 classes, 13 features) and split it
# ------------------------------------------------------------
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ------------------------------------------------------------
# 3. Why scaling matters for k-NN
#    k-NN uses distances, so a feature with a large range (e.g. proline ~ 1000)
#    dominates features with small ranges unless we standardise.
# ------------------------------------------------------------
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_scaled.fit(X_train, y_train)
print(f"k-NN (k=5) without scaling: {knn_raw.score(X_test, y_test):.4f}")
print(f"k-NN (k=5) with scaling:    {knn_scaled.score(X_test, y_test):.4f}")

# ------------------------------------------------------------
# 4. Choose k using 5-fold cross-validation
#    Small k -> low bias, high variance (overfits)
#    Large k -> high bias, low variance (underfits)
# ------------------------------------------------------------
k_values = range(1, 31)
cv_scores, train_scores = [], []
for k in k_values:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_scores.append(cross_val_score(model, X_train, y_train, cv=5).mean())
    train_scores.append(model.fit(X_train, y_train).score(X_train, y_train))

best_k = list(k_values)[int(np.argmax(cv_scores))]
print(f"\nBest k by 5-fold CV = {best_k} (CV accuracy = {max(cv_scores):.4f})")

plt.figure(figsize=(8, 4.5))
plt.plot(list(k_values), train_scores, "o-", label="Training accuracy")
plt.plot(list(k_values), cv_scores, "s-", label="Cross-validation accuracy")
plt.axvline(best_k, color="r", linestyle="--", label=f"Best k = {best_k}")
plt.xlabel("k (number of neighbours)")
plt.ylabel("Accuracy")
plt.title("k-NN: choosing k")
plt.legend()
plt.tight_layout()
plt.savefig("program10_knn_choose_k.png", dpi=120)
plt.show()

# ------------------------------------------------------------
# 5. Compare distance metrics and weighting schemes
#    uniform  - every neighbour gets one vote
#    distance - closer neighbours get larger votes (weight = 1/distance)
# ------------------------------------------------------------
print("\nMetric       Weights    CV accuracy")
for metric in ["euclidean", "manhattan", "chebyshev"]:
    for weights in ["uniform", "distance"]:
        model = make_pipeline(StandardScaler(),
                              KNeighborsClassifier(n_neighbors=best_k, metric=metric, weights=weights))
        score = cross_val_score(model, X_train, y_train, cv=5).mean()
        print(f"{metric:11s}  {weights:9s}  {score:.4f}")

# ------------------------------------------------------------
# 6. Final k-NN model evaluated on the test set
# ------------------------------------------------------------
final_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
final_knn.fit(X_train, y_train)
y_pred = final_knn.predict(X_test)
print(f"\nFinal k-NN test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=wine.target_names))

# ------------------------------------------------------------
# 7. Visualise decision boundaries for different k on 2-D "moons" data
# ------------------------------------------------------------
Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=42)
xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5, 300),
                     np.linspace(Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5, 300))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, k in zip(axes, [1, 5, 15, 50]):
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xm, ym)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(f"k = {k}  (train acc = {clf.score(Xm, ym):.2f})")
plt.suptitle("k-NN decision boundaries: small k overfits, large k underfits")
plt.tight_layout()
plt.savefig("program10_knn_boundaries.png", dpi=120)
plt.show()

# ============================================================
# PART B: KERNEL DENSITY ESTIMATION
# ============================================================
print("=" * 60)
print("PART B: KERNEL DENSITY ESTIMATION")
print("=" * 60)

# ------------------------------------------------------------
# 8. Create 1-D bimodal data (a mixture of two Gaussians)
#    A single Gaussian cannot represent this, but KDE can.
# ------------------------------------------------------------
data_1d = np.concatenate([np.random.normal(-2, 0.8, 300),
                          np.random.normal(3, 1.2, 200)])
x_plot = np.linspace(-6, 8, 500)
true_pdf = 0.6 * norm.pdf(x_plot, -2, 0.8) + 0.4 * norm.pdf(x_plot, 3, 1.2)

# ------------------------------------------------------------
# 9. Effect of bandwidth (h)
#    KDE: p(x) = (1 / (N*h)) * sum K((x - x_i) / h)
#    Small h -> spiky, noisy estimate; large h -> over-smoothed estimate
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(data_1d, bins=40, density=True, alpha=0.3, color="gray", label="Histogram")
axes[0].plot(x_plot, true_pdf, "k--", linewidth=2, label="True density")
for h in [0.1, 0.5, 2.0]:
    kde = KernelDensity(kernel="gaussian", bandwidth=h).fit(data_1d[:, None])
    axes[0].plot(x_plot, np.exp(kde.score_samples(x_plot[:, None])), label=f"h = {h}")
axes[0].set_title("KDE: effect of bandwidth")
axes[0].legend()

# ------------------------------------------------------------
# 10. Effect of kernel shape (with a fixed bandwidth)
# ------------------------------------------------------------
axes[1].plot(x_plot, true_pdf, "k--", linewidth=2, label="True density")
for kernel in ["gaussian", "tophat", "epanechnikov", "exponential"]:
    kde = KernelDensity(kernel=kernel, bandwidth=0.6).fit(data_1d[:, None])
    axes[1].plot(x_plot, np.exp(kde.score_samples(x_plot[:, None])), label=kernel)
axes[1].set_title("KDE: effect of kernel (h = 0.6)")
axes[1].legend()
plt.tight_layout()
plt.savefig("program10_kde_bandwidth_kernel.png", dpi=120)
plt.show()

# ------------------------------------------------------------
# 11. Choose the bandwidth by cross-validation
#     GridSearchCV picks the h that maximises the held-out log-likelihood.
# ------------------------------------------------------------
grid = GridSearchCV(KernelDensity(kernel="gaussian"),
                    {"bandwidth": np.linspace(0.05, 2.0, 40)}, cv=5)
grid.fit(data_1d[:, None])
best_h = grid.best_params_["bandwidth"]
print(f"Best bandwidth by CV = {best_h:.3f}")

# Silverman's rule of thumb, for comparison: h = 1.06 * std * N^(-1/5)
silverman_h = 1.06 * data_1d.std() * len(data_1d) ** (-1 / 5)
print(f"Silverman's rule-of-thumb bandwidth = {silverman_h:.3f}")

best_kde = grid.best_estimator_
plt.figure(figsize=(8, 4.5))
plt.hist(data_1d, bins=40, density=True, alpha=0.3, color="gray", label="Histogram")
plt.plot(x_plot, true_pdf, "k--", linewidth=2, label="True density")
plt.plot(x_plot, np.exp(best_kde.score_samples(x_plot[:, None])), "r",
         label=f"KDE (CV h = {best_h:.2f})")
plt.plot(x_plot, norm.pdf(x_plot, data_1d.mean(), data_1d.std()), "b:",
         label="Single Gaussian (parametric)")
plt.title("KDE with cross-validated bandwidth vs parametric fit")
plt.legend()
plt.tight_layout()
plt.savefig("program10_kde_best.png", dpi=120)
plt.show()

# ============================================================
# PART C: KDE CLASSIFIER AND PERFORMANCE COMPARISON
# ============================================================
print("=" * 60)
print("PART C: PERFORMANCE COMPARISON")
print("=" * 60)

# ------------------------------------------------------------
# 12. A Bayes classifier that uses KDE for each class density
#     P(class | x) ∝ p_KDE(x | class) * P(class)
# ------------------------------------------------------------
class KDEClassifier(BaseEstimator, ClassifierMixin):
    """Bayes generative classifier with a KDE density for each class."""

    def __init__(self, bandwidth=1.0, kernel="gaussian"):
        self.bandwidth = bandwidth
        self.kernel = kernel

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        # Fit one KDE per class
        self.models_ = [KernelDensity(bandwidth=self.bandwidth, kernel=self.kernel).fit(X[y == c])
                        for c in self.classes_]
        # Log prior of each class
        self.log_priors_ = [np.log(np.mean(y == c)) for c in self.classes_]
        return self

    def predict_proba(self, X):
        log_post = np.array([m.score_samples(X) + lp
                             for m, lp in zip(self.models_, self.log_priors_)]).T
        log_post -= log_post.max(axis=1, keepdims=True)    # numerical stability
        probs = np.exp(log_post)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# ------------------------------------------------------------
# 13. Tune the KDE classifier's bandwidth with cross-validation
# ------------------------------------------------------------
kde_clf = GridSearchCV(make_pipeline(StandardScaler(), KDEClassifier()),
                       {"kdeclassifier__bandwidth": np.linspace(0.2, 3.0, 15)}, cv=5)
kde_clf.fit(X_train, y_train)
print(f"Best KDE-classifier bandwidth = {kde_clf.best_params_['kdeclassifier__bandwidth']:.2f}")

# ------------------------------------------------------------
# 14. Compare models: accuracy, CV accuracy, training and prediction time
#     k-NN and KDE are non-parametric (store all the data);
#     Gaussian Naive Bayes is parametric (stores only means and variances).
# ------------------------------------------------------------
models = {
    f"k-NN (k={best_k})": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)),
    "KDE classifier": kde_clf.best_estimator_,
    "Gaussian NB (parametric)": make_pipeline(StandardScaler(), GaussianNB()),
}

names, test_accs, cv_means = [], [], []
print("\nModel                       Test acc   CV acc (±std)     Fit(ms)  Predict(ms)")
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5)
    t0 = time.perf_counter(); model.fit(X_train, y_train); fit_t = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); acc = model.score(X_test, y_test); pred_t = (time.perf_counter() - t0) * 1000
    names.append(name); test_accs.append(acc); cv_means.append(cv.mean())
    print(f"{name:26s}  {acc:.4f}    {cv.mean():.4f} ± {cv.std():.4f}   {fit_t:7.2f}  {pred_t:9.2f}")

# ------------------------------------------------------------
# 15. Bar chart of the comparison
# ------------------------------------------------------------
x_pos = np.arange(len(names))
plt.figure(figsize=(8, 4.5))
plt.bar(x_pos - 0.2, cv_means, width=0.4, label="CV accuracy")
plt.bar(x_pos + 0.2, test_accs, width=0.4, label="Test accuracy")
plt.xticks(x_pos, names)
plt.ylim(0.8, 1.02)
plt.ylabel("Accuracy")
plt.title("Non-parametric vs parametric classifiers (Wine)")
plt.legend()
plt.tight_layout()
plt.savefig("program10_comparison.png", dpi=120)
plt.show()

print("""
Observations:
 - k-NN and KDE make no assumption about the shape of the distribution.
 - Both need the whole training set at prediction time, so prediction is slower
   and memory use grows with the data size.
 - Both depend heavily on one smoothing parameter (k or bandwidth h) and on feature scaling.
 - Gaussian NB is fast and compact but assumes Gaussian, independent features.
""")